# RAG & Knowledge Systems: Retrieval Optimization

Finding documents that are "mathematically similar" isn't always enough to get the right answer. In this notebook, we explore advanced techniques to make our search more intelligent and human-like using **Hugging Face** local embeddings.

## 1. The Retrieval Gap
- **Semantic Mismatch**: A query like "How to fix a bug?" might be similar to "Insect control" if the model isn't specialized.
- **Query Quality**: Users often ask vague questions. RAG works best when the question is descriptive.
- **Noise**: Simple similarity search might return irrelevant parts of a long document.

---

## 2. Environment Setup
We will use LangChain's advanced retriever components with **Hugging Face** embeddings.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

load_dotenv()
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)

# Create a quick DB for testing
docs = [
    Document(page_content="The company policy for working from home requires 2 weeks notice."),
    Document(page_content="Remote workers must use the VPN at all times for security."),
    Document(page_content="Employees can request hardware upgrades every 3 years."),
    Document(page_content="Coffee is provided for free in the main office kitchen.")
]
db = Chroma.from_documents(docs, embeddings)

print("Optimization workspace with Local Embeddings ready!")

## 3. Multi-Query Retrieval
A user might ask "Can I work from my house?". The database might not find a match for "house." Multi-Query uses an LLM to generate multiple versions of the query (e.g., "remote work policy", "home office guidelines") and retrieves documents for all of them.

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever

retriever = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(), 
    llm=llm
)

query = "Can I work from my house?"
optimized_results = retriever.invoke(query)

print(f"Found {len(optimized_results)} relevant documents.")
for doc in optimized_results:
    print(f"- {doc.page_content}")

## 4. Contextual Compression
Sometimes documents are very long, and only one sentence is relevant. Compression uses an LLM to "summarize" or "filter" the retrieved documents before passing them to the final chain, saving tokens.

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=db.as_retriever()
)

compressed_docs = compression_retriever.invoke("What are the VPN rules?")
for doc in compressed_docs:
    print(f"Compressed Content: {doc.page_content}")

## 5. Parent Document Retrieval
This technique stores small chunks for search but retrieves the larger "parent" context (the whole paragraph or section) for the LLM. This provides the AI with more context without overwhelming the search index.

In [ ]:
print("Concept: Small chunks for retrieval, Large chunks for generation.")
# Note: Implementation requires a LocalFileStore for larger persistence, 
# but the principle is clear: fetch the 'Full Document' ID stored in metadata.

## 6. Ensemble Retrieval
Combining **Term Search** (BM25/Keyword) with **Vector Search**. This is the best way to handle technical jargon that might not be in the embedding model's training data.

In [ ]:
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

keyword_retriever = BM25Retriever.from_documents(docs)
keyword_retriever.k = 2

ensemble_retriever = EnsembleRetriever(
    retrievers=[keyword_retriever, db.as_retriever()], 
    weights=[0.5, 0.5]
)

ensemble_results = ensemble_retriever.invoke("home office VPN")
for doc in ensemble_results:
    print(f"Ensemble Result: {doc.page_content}")

## 7. Re-ranking (LLM-Based)
Retrieve many results (e.g., 20), then use a smarter model to rank them from 1 to N based on relevance. This ensures the best answer is at the top even if the vector similarity was low.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

class RankingScore(BaseModel):
    relevance_score: int = Field(description="Score from 1-10 of how relevant the doc is")

def rerank_documents(query, documents):
    reranker_prompt = PromptTemplate.from_template(
        "Score the relevance of the document to the query on a scale of 1-10. Output JSON only.\n"
        "Query: {query}\nDocument: {doc}"
    )
    
    scored_docs = []
    parser = JsonOutputParser(pydantic_object=RankingScore)
    rank_chain = reranker_prompt | llm | parser
    
    print(f"Reranking {len(documents)} docs...")
    for doc in documents:
        score = rank_chain.invoke({"query": query, "doc": doc.page_content})
        scored_docs.append((doc, score['relevance_score']))
    
    # Sort by score descending
    return sorted(scored_docs, key=lambda x: x[1], reverse=True)

results = db.similarity_search("how to fix my computer", k=3)
ranked_results = rerank_documents("computer hardware issues", results)

for doc, score in ranked_results:
    print(f"[Score: {score}] {doc.page_content}")

## 8. Handling Multi-Turn Retrieval (Query Rewriting)
In a conversation, the current query might rely on previous context (e.g., "Tell me more about that"). We use Gemini to rewrite the query into a standalone question.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

def rewrite_query(chat_history, latest_question):
    rewrite_prompt = PromptTemplate.from_template(
        "Given the chat history and the latest question, rewrite it into a standalone search query.\n"
        "History: {history}\nQuestion: {question}\nStandalone Query:"
    )
    chain = rewrite_prompt | llm
    res = chain.invoke({"history": chat_history, "question": latest_question})
    return res.content

history = "User: What is the VPN policy?\nAI: You must use it for security at all times."
question = "Is that true for home office too?"

standalone_query = rewrite_query(history, question)
print(f"Latest Question: {question}")
print(f"Rewritten Query: {standalone_query}")

## 9. Measuring Performance (LLM-Grounded Evaluation)
How do we know RAG is working? We use the LLM as a judge to check if the retrieved context actually contains the answer and if the final response is faithful to that context.

In [ ]:
class RelevanceGrade(BaseModel):
    is_relevant: bool = Field(description="True if the context helps answer the question")

def evaluate_rag(query, context, answer):
    eval_prompt = PromptTemplate.from_template(
        "Analyze the RAG output.\n"
        "Question: {query}\nContext: {context}\nAnswer: {answer}\n"
        "Is the answer supported by the context and relevant to the question? Output JSON only."
    )
    
    parser = JsonOutputParser(pydantic_object=RelevanceGrade)
    eval_chain = eval_prompt | llm | parser
    
    return eval_chain.invoke({"query": query, "context": context, "answer": answer})

context = "Policy: Employees get 20 vacation days."
answer = "You have 20 vacation days per year."
check = evaluate_rag("How many days of leave do I have?", context, answer)
print(f"Evaluation results: {check}")

## 10. Summary & Pro-Tips
1. **Combine Local & Cloud**: Use local Hugging Face embeddings for speed/cost, and Gemini for the complex reasoning task of compression/multi-query.
2. **Multi-Query is a Life Saver**: It fixes bad user questions before they reach the database.
3. **Evaluate with LLMs**: Writing code to test RAG manually is hard. Using an LLM as a grader is the industry standard for rapid prototyping.

---